# Ev tapşırığı — Regression (Simple / Multiple / Polynomial)

**Modul 3 · Dərs 1**

Bu notebook `data/bal_10.csv`, `data/bal_20.csv` və `data/bal_50.csv` fayllarını oxuyur. Hər faylda üç feature-dən istifadə olunur: `cicek_say`, `temperatur`, `rutubet`; hədəf sütunu `bal_kq`-dır. Kod nümunələri dərsdəki notebook-dan köçürülməyib.

> Notebook-u `Homework3` qovluğundan və ya onun bir üst qovluğundan açıb işlədin. CSV faylları `Homework3/data/` qovluğunda olmalıdır.

## 1. Kitabxanalar və verilənlərin yoxlanması

Bu tapşırıqda ölçülər və əmsallar aydın görünsün deyə hər iki model bütün müşahidələr üzərində öyrədilir. Buna görə aşağıdakı metriklər **öyrədilmə verilənləri** üzrədir; onlar yeni, görünməmiş məlumatlarda modelin keyfiyyətini müstəqil ölçmür.

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import PolynomialFeatures

FEATURES = ["cicek_say", "temperatur", "rutubet"]
TARGET = "bal_kq"
FILES = {"bal_10": 10, "bal_20": 20, "bal_50": 50}
NEW_DAY = pd.DataFrame([[180, 26, 55]], columns=FEATURES)

data_candidates = [
    Path.cwd() / "data",
    Path.cwd() / "Homework3" / "data",
    Path.cwd().parent / "Homework3" / "data",
]
DATA_DIR = next((path for path in data_candidates if path.is_dir()), data_candidates[0])

datasets = {}
for name, expected_rows in FILES.items():
    csv_path = DATA_DIR / f"{name}.csv"
    if not csv_path.is_file():
        raise FileNotFoundError(
            f"{csv_path} tapılmadı. Üç CSV faylını Homework3/data/ qovluğuna yerləşdirin."
        )

    data = pd.read_csv(csv_path)
    required_columns = FEATURES + [TARGET]
    missing_columns = [column for column in required_columns if column not in data.columns]
    if missing_columns:
        raise ValueError(f"{csv_path} faylında tələb olunan sütunlar yoxdur: {missing_columns}")
    if len(data) < 2:
        raise ValueError(f"{csv_path} model qurmaq üçün ən azı 2 sətir saxlamalıdır.")

    datasets[name] = data
    if len(data) != expected_rows:
        warnings.warn(
            f"{name}.csv üçün {expected_rows} sətir gözlənilirdi, faktiki {len(data)} sətirdir."
        )
    print(f"{name}.csv: {len(data)} sətir; sütunlar: {', '.join(data.columns)}")

display(datasets["bal_10"].head())

## 2. Multiple Linear Regression

Hər CSV üçün üç seçilmiş feature-dən istifadə edərək xətti model qurulur. Çap olunan `w` qiymətləri feature-lərin əmsalları, `b` isə sərbəst həddir.

In [ ]:
linear_models = {}
linear_predictions = {}
for name, data in datasets.items():
    X = data[FEATURES]
    y = data[TARGET]

    model = LinearRegression()
    model.fit(X, y)
    linear_models[name] = model
    linear_predictions[name] = model.predict(X)

    print(f"\n{name} — Multiple Linear Regression")
    for feature, weight in zip(FEATURES, model.coef_):
        print(f"w[{feature}] = {weight:.6f}")
    print(f"b = {model.intercept_:.6f}")
    print(f"Yeni gün üçün proqnoz (180 çiçək, 26°C, 55%): {model.predict(NEW_DAY)[0]:.4f} kq")

## 3. Polynomial Regression (dərəcə 2)

`PolynomialFeatures` ilkin feature-lərdən ikinci dərəcəli qarşılıqlı təsir və kvadrat terminlər yaradır. `include_bias=False` seçildiyi üçün modelin sərbəst həddi polynomial transformer tərəfindən təkrar əlavə edilmir.

In [ ]:
polynomial_models = {}
polynomial_transformers = {}
polynomial_predictions = {}
polynomial_feature_names = {}

for name, data in datasets.items():
    X = data[FEATURES]
    y = data[TARGET]

    transformer = PolynomialFeatures(degree=2, include_bias=False)
    X_polynomial = transformer.fit_transform(X)
    feature_names = transformer.get_feature_names_out(FEATURES)

    model = LinearRegression()
    model.fit(X_polynomial, y)
    new_day_polynomial = transformer.transform(NEW_DAY)

    polynomial_transformers[name] = transformer
    polynomial_models[name] = model
    polynomial_predictions[name] = model.predict(X_polynomial)
    polynomial_feature_names[name] = feature_names

    print(f"\n{name} — Polynomial Regression (dərəcə 2)")
    for feature, weight in zip(feature_names, model.coef_):
        print(f"{feature}: {weight:.6f}")
    print(f"b = {model.intercept_:.6f}")
    print(f"Yeni gün üçün proqnoz (180 çiçək, 26°C, 55%): {model.predict(new_day_polynomial)[0]:.4f} kq")

## 4. MAE, MSE və R² cədvəli

MAE və MSE nə qədər kiçikdirsə, proqnoz xətası bir o qədər azdır. R² nə qədər böyükdürsə, model öyrədilmə məlumatındakı dəyişkənliyi bir o qədər yaxşı izah edir. Aşağıdakı dəyərlər in-sample, yəni modelin öyrədildiyi həmin sətirlər üzərində hesablanıb.

In [ ]:
metric_rows = []
for name, data in datasets.items():
    actual = data[TARGET]
    model_outputs = [
        ("Multiple Linear", linear_predictions[name]),
        ("Polynomial (dərəcə 2)", polynomial_predictions[name]),
    ]

    for model_name, predicted in model_outputs:
        metric_rows.append(
            {
                "Fayl": name,
                "Sətir sayı": len(data),
                "Model": model_name,
                "MAE": mean_absolute_error(actual, predicted),
                "MSE": mean_squared_error(actual, predicted),
                "R²": r2_score(actual, predicted),
            }
        )

metrics_table = pd.DataFrame(metric_rows)
display(metrics_table.round({"MAE": 4, "MSE": 4, "R²": 4}))

## 5. Qısa nəticə

Aşağıdakı xülasə metrik cədvəlindən avtomatik qurulur. Müqayisə eyni CSV daxilində aparılır; müxtəlif ölçülü fayllar arasındakı fərqlər isə ehtiyatla şərh edilməlidir.

In [ ]:
for name, data in datasets.items():
    subset = metrics_table[metrics_table["Fayl"] == name].set_index("Model")
    linear_row = subset.loc["Multiple Linear"]
    polynomial_row = subset.loc["Polynomial (dərəcə 2)"]

    def winner(metric, lower_is_better):
        linear_value = linear_row[metric]
        polynomial_value = polynomial_row[metric]
        if np.isclose(linear_value, polynomial_value):
            return "bərabər"
        linear_wins = linear_value < polynomial_value if lower_is_better else linear_value > polynomial_value
        return "Multiple Linear" if linear_wins else "Polynomial"

    print(f"{name} ({len(data)} sətir):")
    print(f"  MAE üzrə daha yaxşı: {winner('MAE', lower_is_better=True)}")
    print(f"  MSE üzrə daha yaxşı: {winner('MSE', lower_is_better=True)}")
    print(f"  R² üzrə daha yaxşı: {winner('R²', lower_is_better=False)}")
    print(
        f"  Dəyərlər — MAE: xətti {linear_row['MAE']:.4f}, polynomial {polynomial_row['MAE']:.4f}; "
        f"MSE: xətti {linear_row['MSE']:.4f}, polynomial {polynomial_row['MSE']:.4f}; "
        f"R²: xətti {linear_row['R²']:.4f}, polynomial {polynomial_row['R²']:.4f}."
    )

print(
    "\nNəticə: sətir sayının artması daha çox müşahidə təqdim edir və model əmsallarının "
    "öyrədilməsinə kömək edə bilər. Lakin fərqli faylların metrikləri müxtəlif nümunələrdə "
    "hesablandığından, sətir sayının təsirini yalnız bu cədvəllə səbəb-nəticə kimi sübut etmək "
    "olmaz. Polynomial model daha mürəkkəb əlaqələri tuta bildiyi üçün öyrədilmə MAE-si aşağı "
    "ola bilər, amma az sətirdə həddindən artıq öyrənmə riski artır. Buna görə polynomial "
    "model həmişə daha yaxşı deyil; etibarlı ümumiləşdirmə müqayisəsi üçün ayrıca test dəsti "
    "və ya çarpaz yoxlama istifadə edilməlidir."
)